In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

# Load training data
df = pd.read_csv('cs-training.csv')
df.head()


In [ ]:
# Remove duplicate records and the ID column
df = df.drop_duplicates()
df = df.drop(columns=['Unnamed: 0'])
df.head()


In [ ]:
# Check missing values
df.isnull().sum()


In [ ]:
# Check data types and non-null counts
df.info()


In [ ]:
# Summary statistics
df.describe()


In [ ]:
# Remove unreasonable age values and keep borrowers between 18 and 85
df = df[(df['age'] >= 18) & (df['age'] <= 85)]
df.describe()


In [ ]:
# Remove abnormal 96/98 codes in delinquency count variables
print(df['NumberOfTime30-59DaysPastDueNotWorse'].unique(), '')
print(df['NumberOfTime60-89DaysPastDueNotWorse'].unique(), '')
print(df['NumberOfTimes90DaysLate'].unique(), '')

late_cols = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate'
]

outlier_values = [96, 98]
for col in late_cols:
    df = df[~df[col].isin(outlier_values)]

df = df.copy()
df.describe()


In [ ]:
# Inspect variables with extreme values before applying capping
# Capping thresholds should be learned from the training set after train/validation split
df.describe()


In [ ]:
# Plot sampled histograms and boxplots for exploratory data analysis
plot_df = df.sample(n=30000, random_state=42)

select_index = plot_df.select_dtypes(include=['float64', 'int64']).columns
n_col = 4
n_row = math.ceil(len(select_index) / n_col)

plt.figure(figsize=(29, 20))
for i, col in enumerate(select_index):
    plt.subplot(n_row, n_col, i + 1)
    sns.histplot(x=plot_df[col], bins=50, kde=False, color='red')
    plt.title(f'{col} Distribution')
plt.tight_layout()
plt.show()

plt.figure(figsize=(39, 30))
for i, col in enumerate(select_index):
    plt.subplot(n_row, n_col, i + 1)
    sns.boxplot(y=plot_df[col], color='skyblue')
    plt.title(f'{col} Outliers')
plt.tight_layout()
plt.show()


In [ ]:
# Plot correlation heatmap to inspect multicollinearity
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()


In [ ]:
# Create a copy for the WOE-Logistic modeling workflow
df_lin = df.copy()
df_lin.describe()
df_lin.isna().sum()


In [ ]:
# Fill missing values with zero for variables where missingness can be interpreted as no count
cols_fill_zero = [
    'NumberOfDependents',
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
    'NumberOfTime60-89DaysPastDueNotWorse'
]
df_lin[cols_fill_zero] = df_lin[cols_fill_zero].fillna(0)
df_lin.isna().sum()


In [ ]:
# Split the data into training and validation sets
y = df_lin['SeriousDlqin2yrs']
X = df_lin.drop(columns=['SeriousDlqin2yrs'])
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = X_train.copy()
X_val = X_val.copy()

# Impute MonthlyIncome using the median learned from the training set only
income_median = X_train['MonthlyIncome'].median()
X_train['MonthlyIncome'] = X_train['MonthlyIncome'].fillna(income_median)
X_val['MonthlyIncome'] = X_val['MonthlyIncome'].fillna(income_median)

# Cap selected continuous variables at the 99th percentile learned from the training set only
cap_cols = [
    'RevolvingUtilizationOfUnsecuredLines',
    'DebtRatio',
    'MonthlyIncome'
]

cap_values = {}
for col in cap_cols:
    cap_values[col] = X_train[col].quantile(0.99)
    X_train[col] = X_train[col].clip(upper=cap_values[col])
    X_val[col] = X_val[col].clip(upper=cap_values[col])

# Apply log transformation to reduce the impact of the long-tailed income distribution
X_train['MonthlyIncome_Log'] = np.log1p(X_train['MonthlyIncome'])
X_val['MonthlyIncome_Log'] = np.log1p(X_val['MonthlyIncome'])


In [ ]:
# Bivariate analysis for binning decisions after train/validation split
df_train = X_train.copy()
df_train['SeriousDlqin2yrs'] = y_train
target = 'SeriousDlqin2yrs'
features = df_train.drop(columns=[target]).select_dtypes(include=['float64', 'int64']).columns
plt.figure(figsize=(15, 12))

for i, col in enumerate(features):
    plt.subplot(4, 3, i + 1)
    
    # For low-cardinality variables, plot default rate by value
    if df_train[col].nunique() < 10:
        sns.barplot(x=col, y=target, data=df_train, errorbar=None, palette='Reds')
        plt.title(f'{col}: Default Rate (Higher is Worse)', color='red')
        plt.ylabel('Probability of Default')
    else:
        # For continuous variables, compare distributions between good and bad borrowers
        sns.boxplot(x=target, y=col, data=df_train, showfliers=False, palette='Set2')
        plt.title(f'{col} Distribution by Target', color='blue')

plt.tight_layout()
plt.show()


In [ ]:
# Decision-tree-based binning function
def tree_binning(x, y, max_leaf_nodes=4):
    """
    Use a decision tree to find cut points for a one-dimensional variable.
    The output is a list of bin edges such as [-inf, 25, 40, 60, inf].
    """
    x = x.values.reshape(-1, 1)
    
    clf = DecisionTreeClassifier(
        max_leaf_nodes=max_leaf_nodes,
        min_samples_leaf=0.05
    )
    clf.fit(x, y)
    
    thresholds = clf.tree_.threshold
    thresholds = thresholds[thresholds != -2]
    bins = [-np.inf] + sorted(thresholds) + [np.inf]
    
    return bins


In [ ]:
# Learn bin cut points from the training set only
bin_dict = {}

bin_dict['age'] = tree_binning(X_train['age'], y_train)
bin_dict['rev'] = tree_binning(X_train['RevolvingUtilizationOfUnsecuredLines'], y_train)
bin_dict['debt'] = tree_binning(X_train['DebtRatio'], y_train)
bin_dict['income'] = tree_binning(X_train['MonthlyIncome_Log'], y_train)
bin_dict['open_credit'] = tree_binning(X_train['NumberOfOpenCreditLinesAndLoans'], y_train)
bin_dict['real_estate'] = tree_binning(X_train['NumberRealEstateLoansOrLines'], y_train)


In [ ]:
# Apply continuous-variable and business-based binning rules
def process_binning(df_input, bin_dict):
    df = df_input.copy()
    
    # Continuous variables: decision-tree-based bins
    df['Age'] = pd.cut(df['age'], bins=bin_dict['age'])
    df['Rev'] = pd.cut(df['RevolvingUtilizationOfUnsecuredLines'], bins=bin_dict['rev'])
    df['DebtRatio_Bin'] = pd.cut(df['DebtRatio'], bins=bin_dict['debt'])
    df['Income_Bin'] = pd.cut(df['MonthlyIncome_Log'], bins=bin_dict['income'])
    df['OpenCredit_Bin'] = pd.cut(df['NumberOfOpenCreditLinesAndLoans'], bins=bin_dict['open_credit'])
    df['RealEstate_Bin'] = pd.cut(df['NumberRealEstateLoansOrLines'], bins=bin_dict['real_estate'])
    
    # Delinquency count variables: business-based bins
    df['30-59Days_Bin'] = pd.cut(
        df['NumberOfTime30-59DaysPastDueNotWorse'],
        bins=[-1, 0, 1, 2, 100],
        labels=['Never_30_59', 'Once_30_59', 'Twice_30_59', 'More_30_59']
    )
    df['60-89Days_Bin'] = pd.cut(
        df['NumberOfTime60-89DaysPastDueNotWorse'],
        bins=[-1, 0, 1, 100],
        labels=['Never_60_89', 'Once_60_89', 'More_60_89']
    )
    df['90Days_Bin'] = pd.cut(
        df['NumberOfTimes90DaysLate'],
        bins=[-1, 0, 1, 100],
        labels=['Never_90', 'Once_90', 'More_90']
    )
    df['Dependents_Bin'] = pd.cut(
        df['NumberOfDependents'],
        bins=[-1, 0, 1, 2, 100],
        labels=['No_Dep', 'One_Dep', 'Two_Dep', 'More_Dep']
    )

    return df


In [ ]:
# WOE and IV calculation function
def calculate_woe_iv(df, feature, target):
    # Count total and bad borrowers in each bin
    grouped = df.groupby(feature, observed=True)[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    
    # Calculate good borrowers and bad rate
    grouped['good'] = grouped['total'] - grouped['bad']
    grouped['bad_rate'] = grouped['bad'] / grouped['total']
    
    # Calculate overall good and bad totals
    total_good = grouped['good'].sum()
    total_bad = grouped['bad'].sum()
    
    # Calculate good and bad distributions by bin
    grouped['good_pct'] = grouped['good'] / total_good
    grouped['bad_pct'] = grouped['bad'] / total_bad
    
    # Calculate WOE; the small constant avoids log(0)
    grouped['WOE'] = np.log(
        (grouped['good_pct'] + 1e-6) /
        (grouped['bad_pct'] + 1e-6)
    )
    
    # Calculate IV contribution by bin
    grouped['IV_component'] = (
        grouped['good_pct'] - grouped['bad_pct']
    ) * grouped['WOE']
    
    # Sum IV contribution to obtain feature-level IV
    iv = grouped['IV_component'].sum()
    
    return grouped, iv


In [ ]:
# Variables to be transformed by WOE
bin_cols = [
    'Age', 'Rev',
    'DebtRatio_Bin', 'Income_Bin',
    'OpenCredit_Bin', 'RealEstate_Bin',
    '30-59Days_Bin', '60-89Days_Bin', '90Days_Bin',
    'Dependents_Bin'
]

# Apply binning to the training set and attach target
df_train_bin = process_binning(X_train, bin_dict)
df_train_bin['target'] = y_train

# Store WOE mappings, IV summary, and detailed WOE tables
woe_maps = {}
iv_summary = {}
woe_detail_list = []

for col in bin_cols:
    # Calculate WOE table and IV for the current variable
    woe_table, iv_value = calculate_woe_iv(df_train_bin, col, 'target')
    
    # Build a clean long-format table for reporting
    woe_table_report = woe_table.reset_index().rename(columns={col: 'bin'})
    woe_table_report['feature'] = col
    woe_table_report['IV'] = iv_value
    
    report_cols = [
        'feature', 'bin', 'total', 'bad', 'good', 'bad_rate',
        'good_pct', 'bad_pct', 'WOE', 'IV_component', 'IV'
    ]
    woe_table_report = woe_table_report[report_cols]
    woe_detail_list.append(woe_table_report)
    
    # Save bin-to-WOE mapping for model input transformation
    woe_maps[col] = woe_table['WOE'].to_dict()
    iv_summary[col] = iv_value

# Combine all WOE detail tables
woe_detail_df = pd.concat(woe_detail_list, ignore_index=True)

# Create IV ranking table
iv_df = pd.DataFrame({
    'feature': list(iv_summary.keys()),
    'IV': list(iv_summary.values())
})
iv_df = iv_df.sort_values(by='IV', ascending=False)

def iv_strength(iv):
    if iv < 0.02:
        return 'useless'
    elif iv < 0.1:
        return 'weak'
    elif iv < 0.3:
        return 'medium'
    elif iv < 0.5:
        return 'strong'
    else:
        return 'very strong'

iv_df['strength'] = iv_df['IV'].apply(iv_strength)

print(iv_df)

# Display WOE details for the top five IV variables
top_iv_features = iv_df.head(5)['feature'].tolist()
woe_detail_df[woe_detail_df['feature'].isin(top_iv_features)]


In [ ]:
# Apply WOE transformation to training and validation sets
# Unknown bins are assigned WOE=0, representing average risk contribution
def apply_woe(df, woe_maps, default_woe=0):
    df = df.copy()
    
    for col, mapping in woe_maps.items():
        mapped_woe = df[col].map(mapping).astype('float64')
        df[col] = mapped_woe.fillna(default_woe)
    
    return df


In [ ]:
# Apply binning
X_train_bin = process_binning(X_train, bin_dict)
X_val_bin = process_binning(X_val, bin_dict)

# Apply WOE transformation
X_train_woe = apply_woe(X_train_bin, woe_maps)
X_val_woe = apply_woe(X_val_bin, woe_maps)

# Drop raw variables after WOE transformation to avoid duplicated information
drop_cols = [
    'age', 'RevolvingUtilizationOfUnsecuredLines',
    'DebtRatio', 'MonthlyIncome', 'MonthlyIncome_Log',
    'NumberOfOpenCreditLinesAndLoans', 'NumberRealEstateLoansOrLines',
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfDependents'
]

X_train_woe = X_train_woe.drop(columns=drop_cols)
X_val_woe = X_val_woe.drop(columns=drop_cols)

# Select variables with IV >= 0.02
selected_features = iv_df.loc[iv_df['IV'] >= 0.02, 'feature'].tolist()

X_train_woe = X_train_woe[selected_features]
X_val_woe = X_val_woe[selected_features]

print('Train WOE missing values:', X_train_woe.isna().sum().sum())
print('Validation WOE missing values:', X_val_woe.isna().sum().sum())


In [ ]:
# Train Logistic Regression using selected WOE-transformed variables
model = LogisticRegression(
    solver='liblinear',
    penalty='l2',
    C=1
)

model.fit(X_train_woe, y_train)

# Predict validation-set probability of default
# y_pro represents the predicted probability of serious delinquency within two years
y_pro = model.predict_proba(X_val_woe)[:, 1]


In [ ]:
# Model evaluation functions
# AUC measures overall ranking ability
# KS measures the maximum separation between good and bad borrowers
# Gini is calculated as 2 * AUC - 1

def compute_ks(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    ks = max(tpr - fpr)
    return ks


def evaluate_binary_model(y_true, y_prob, model_name):
    auc = roc_auc_score(y_true, y_prob)
    ks = compute_ks(y_true, y_prob)
    gini = 2 * auc - 1
    
    return {
        'model': model_name,
        'AUC': auc,
        'KS': ks,
        'Gini': gini
    }

# Summarize model performance for reporting
model_metrics = pd.DataFrame([
    evaluate_binary_model(y_val, y_pro, 'Logistic WOE Scorecard')
])

model_metrics


In [ ]:
# Decile and Lift analysis
# Sort borrowers by predicted default probability and divide them into ten equal-sized groups
def decile_analysis(y_true, y_prob):
    decile_df = pd.DataFrame({
        'target': y_true,
        'pred_prob': y_prob
    })
    
    decile_df = decile_df.sort_values('pred_prob', ascending=False).reset_index(drop=True)
    decile_df['decile'] = pd.qcut(
        decile_df.index,
        q=10,
        labels=range(1, 11)
    )
    
    decile_table = decile_df.groupby('decile', observed=True).agg(
        total=('target', 'count'),
        bad=('target', 'sum'),
        avg_pred_prob=('pred_prob', 'mean'),
        bad_rate=('target', 'mean')
    ).reset_index()
    
    decile_table['cum_bad'] = decile_table['bad'].cumsum()
    decile_table['cum_bad_capture_rate'] = decile_table['cum_bad'] / decile_table['bad'].sum()
    
    overall_bad_rate = decile_df['target'].mean()
    decile_table['lift'] = decile_table['bad_rate'] / overall_bad_rate
    
    return decile_table

# Output decile table for ranking performance analysis
decile_table = decile_analysis(y_val, y_pro)
decile_table


In [ ]:
# Plot ROC curve
auc_value = model_metrics.loc[0, 'AUC']
fpr, tpr, thresholds = roc_curve(y_val, y_pro)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'AUC={auc_value:.2f}')
plt.plot([0, 1], [0, 1], 'r--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Logistic WOE Scorecard ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Cutoff strategy analysis
# Simulate business impact under different rejection rates
def cutoff_strategy(y_true, y_prob, reject_rate):
    strategy_df = pd.DataFrame({
        'target': y_true,
        'pred_prob': y_prob
    })
    
    strategy_df = strategy_df.sort_values('pred_prob', ascending=False).reset_index(drop=True)
    reject_n = int(len(strategy_df) * reject_rate)
    
    rejected = strategy_df.iloc[:reject_n]
    approved = strategy_df.iloc[reject_n:]
    
    return {
        'reject_rate': reject_rate,
        'approval_rate': 1 - reject_rate,
        'bad_capture_rate': rejected['target'].sum() / strategy_df['target'].sum(),
        'approved_bad_rate': approved['target'].mean(),
        'rejected_bad_rate': rejected['target'].mean()
    }

# Compare strategy outcomes at 10%, 20%, and 30% rejection rates
strategy_table = pd.DataFrame([
    cutoff_strategy(y_val, y_pro, 0.10),
    cutoff_strategy(y_val, y_pro, 0.20),
    cutoff_strategy(y_val, y_pro, 0.30)
])

strategy_table
